# Exploration notebook

Thin query layer: calls `load.py` and `plot.py` only.
No analysis logic lives in cells.

Set `P.SAVE_FIGURES = True` to write PDFs instead of displaying inline.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import analysis.plot as P
from analysis.load import (
    load_results, filter_results, to_dataframe, aggregate_folds,
    load_classification, load_groups,
)

P.SAVE_FIGURES = False
OUTPUT_DIR = 'figures/'

## Browse available results

In [ ]:
results = load_results('../results/')
df      = to_dataframe(results)
print(f'{len(results)} experiments loaded')
df.head()

In [ ]:
# What parameter combinations have been run?
PARAMS = [c for c in df.columns if c not in ('timestamp', '_path')]
df[PARAMS].drop_duplicates().sort_values(PARAMS[:2] if len(PARAMS) >= 2 else PARAMS)

## Classification results — k-fold

In [ ]:
KFOLD_DIR = '../results/study_classification_kfold_gamma1.0_YYYYMMDD_HHMMSS/'  # adjust
kfold_results = load_results(KFOLD_DIR)
_, _, metadata = load_classification('../results/classification_dataset/')
idx_to_label  = metadata['idx_to_label']

In [ ]:
# Confusion matrices for the first fold
fold0 = filter_results(kfold_results)
by_method = {r['method']: r for r in fold0 if 'method' in r}
figs = P.plot_confusion_matrices_all_methods(by_method, idx_to_label, OUTPUT_DIR)

In [ ]:
# Aggregated F1 per method across folds
agg = aggregate_folds(kfold_results, group_by=['method'])
for (method,), metrics in agg.items():
    vals = metrics.get('f1_weighted', [])
    if vals:
        print(f'{method:25s}  F1={np.mean(vals):.4f} ± {np.std(vals):.4f}')

## Gamma sensitivity

In [ ]:
GAMMA_DIR = '../results/study_classification_gamma_YYYYMMDD_HHMMSS/'  # adjust
gamma_results = load_results(GAMMA_DIR)
raw_agg = aggregate_folds(gamma_results, group_by=['gamma', 'method'])

by_method = {}
for (gamma, method), metrics in raw_agg.items():
    if method:
        by_method.setdefault(method, {})[gamma] = metrics.get('f1_weighted', [])

fig = P.plot_gamma_sensitivity(by_method, output_dir=OUTPUT_DIR)
plt.show()

## Sample size sensitivity

In [ ]:
SAMPLES_DIR = '../results/study_classification_samples_gamma1.0_YYYYMMDD_HHMMSS/'  # adjust
samples_results = load_results(SAMPLES_DIR)
raw_agg = aggregate_folds(samples_results, group_by=['n_train', 'method'])

by_method = {}
for (n_train, method), metrics in raw_agg.items():
    if method:
        by_method.setdefault(method, {})[n_train] = metrics.get('f1_weighted', [])

fig = P.plot_sample_size_sensitivity(by_method, output_dir=OUTPUT_DIR)
plt.show()

## Barycenter RMSE

In [ ]:
RMSE_DIR = '../results/study_barycenter_rmse_YYYYMMDD_HHMMSS/'  # adjust
rmse_results = load_results(RMSE_DIR)

aggregated = {
    (r['n_samples'], r['estimator']): {
        'mean_rmse_euclidean':   r['mean_rmse_euclidean'],
        'std_rmse_euclidean':    r['std_rmse_euclidean'],
        'mean_rmse_wasserstein': r['mean_rmse_wasserstein'],
        'std_rmse_wasserstein':  r['std_rmse_wasserstein'],
    }
    for r in rmse_results
    if 'n_samples' in r and 'estimator' in r
}

fig = P.plot_rmse_vs_samples(aggregated, output_dir=OUTPUT_DIR)
plt.show()

## Interpolation

In [ ]:
INTERP_DIR = '../results/study_interpolation_gamma1.0_YYYYMMDD_HHMMSS/'  # adjust
interp_results = load_results(INTERP_DIR)

for r in interp_results[:1]:  # show first result
    method = r.get('method', 'euclidean_params')
    la, lb = r.get('label_a'), r.get('label_b')
    fig = P.plot_interpolation_sequence(
        r[f'interpolants_{method}'],
        r['t_values'],
        label_a=idx_to_label.get(la, str(la)),
        label_b=idx_to_label.get(lb, str(lb)),
        method=method,
    )
    plt.show()

## Class barycenters

In [ ]:
from src.dataloader import load_classification as src_load, preprocess_samples
X_arr, Y_arr, _ = src_load('../results/classification_dataset/')
X_list = preprocess_samples(X_arr)

# Extract barycenters for one method from fold 0
fold0_method = filter_results(kfold_results, method='euclidean_params')[0]
barycenters = {int(k.replace('barycenter_', '')): fold0_method[k]
               for k in fold0_method if k.startswith('barycenter_')
               and isinstance(fold0_method[k], np.ndarray)}

P.plot_class_barycenters(barycenters, X_list, Y_arr, idx_to_label,
                          method_name='euclidean_params', output_dir=OUTPUT_DIR)